# Sample 2 Completed-Cycle Pipeline: Step-by-Step Audit

This notebook executes the current `src_2` pipeline one stage at a time and
records the output of every completed stage in one Excel workbook.

**Business objective:** use the completed batch to score campaigns and their
adsets, ads, creatives, and audiences from observed WhatsApp business outcomes,
then create an illustrative next-cycle budget scenario. Meta metrics explain
delivery behavior; they do not replace delivered-order outcomes.

The notebook keeps Meta media, WhatsApp conversations, and order lines at their
natural grains. It aggregates them separately before joining scorecards, which
prevents duplicated spend or duplicated orders.

The LLM section is disabled by default to avoid unplanned API calls. Set
`RUN_LLM = True` in the setup cell to execute the 12 campaign analyses, one
portfolio synthesis, and one stakeholder report call.

## Pipeline map

1. Read the three Sample 2 JSON files.
2. Normalize dimensions and privacy-safe fact tables.
3. Assess Meta-to-WhatsApp data quality.
4. Load and validate campaign-type and budget policies.
5. Aggregate five scorecard levels and calculate KPIs.
6. Document the derived KPI formulas.
7. Build campaign evidence packs and benchmarks.
8. Assign deterministic campaign target and funding decisions.
9. Propagate decisions to adsets, ads, creatives, and audiences.
10. Build the illustrative campaign-level budget scenario.
11. Optionally run the structured LLM narrative pipeline.
12. Assemble final audit outputs.

**Known current gaps:** organic/direct outcomes are retained but not surfaced as
a formal baseline; daily rows are rolled up but trend/fatigue features are not
calculated; historical benchmarks are not yet available; child entities receive
actions but not explicit budget units; the test envelope follows historical
experimental spend instead of a fixed 30% reserve.

In [1]:
# Step 0 - Runtime setup and Excel audit logger
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

ROOT = Path.cwd().resolve()
if not (ROOT / "src_2").exists():
    raise RuntimeError("Run this notebook from the Team2-MarketingExpert repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from src_2.paths import INPUT_DIR

INPUT_DIRECTORY = Path(INPUT_DIR)
OUTPUT_DIRECTORY = ROOT / "outputs" / "sample2_pipeline_audit"
WORKBOOK_PATH = OUTPUT_DIRECTORY / "sample2_pipeline_step_log.xlsx"
FINAL_JSON_PATH = OUTPUT_DIRECTORY / "sample2_completed_cycle_report.json"
RUN_LLM = False
MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
if WORKBOOK_PATH.exists():
    WORKBOOK_PATH.unlink()

SHEET_LOG: dict[str, dict] = {
    "00_Run_Log": {
        "sheet_sequence": 1,
        "worksheet_name": "00_Run_Log",
        "pipeline_step": "00_runtime_setup",
        "data_rows": 0,
        "data_columns": 7,
        "purpose": "Index of every physical worksheet written to this workbook",
        "last_written_utc": None,
    }
}


def _excel_value(value):
    if isinstance(value, (dict, list, tuple, set)):
        value = json.dumps(value, ensure_ascii=True, default=str)
    if pd.isna(value) if not isinstance(value, str) else False:
        return None
    text = str(value) if not isinstance(value, (int, float, bool, datetime)) else value
    return text[:32000] if isinstance(text, str) else text


def _excel_frame(value) -> pd.DataFrame:
    if isinstance(value, pd.Series):
        frame = value.to_frame().reset_index()
    elif isinstance(value, pd.DataFrame):
        frame = value.copy()
    elif isinstance(value, dict):
        frame = pd.DataFrame([value])
    else:
        frame = pd.DataFrame(value)
    frame.columns = [str(column) for column in frame.columns]
    for column in frame.columns:
        if isinstance(frame[column].dtype, pd.DatetimeTZDtype):
            frame[column] = frame[column].dt.tz_convert(None)
        elif frame[column].dtype == "object":
            frame[column] = frame[column].map(_excel_value)
    return frame.replace([float("inf"), float("-inf")], None)


def _style_sheets(sheet_names: list[str]) -> None:
    workbook = load_workbook(WORKBOOK_PATH)
    header_fill = PatternFill("solid", fgColor="17365D")
    header_font = Font(color="FFFFFF", bold=True)
    for sheet_name in sheet_names:
        worksheet = workbook[sheet_name]
        worksheet.sheet_view.showGridLines = False
        worksheet.freeze_panes = "A2"
        if worksheet.max_row >= 1 and worksheet.max_column >= 1:
            worksheet.auto_filter.ref = worksheet.dimensions
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(vertical="center", wrap_text=True)
        for column_index in range(1, worksheet.max_column + 1):
            header = str(worksheet.cell(1, column_index).value or "").lower()
            values = [worksheet.cell(row, column_index).value for row in range(1, min(worksheet.max_row, 250) + 1)]
            width = min(max(len(str(value)) for value in values if value is not None) + 2, 42) if any(value is not None for value in values) else 12
            worksheet.column_dimensions[get_column_letter(column_index)].width = max(width, 10)
            for row in range(2, worksheet.max_row + 1):
                cell = worksheet.cell(row, column_index)
                cell.alignment = Alignment(vertical="top", wrap_text=False)
                if any(token in header for token in ("date", "timestamp", "extracted_at")):
                    cell.number_format = "yyyy-mm-dd hh:mm"
                elif any(token in header for token in ("spend", "revenue", "value", "budget_units", "benchmark", "actual")):
                    cell.number_format = "#,##0.00"
                elif header.endswith("_rate") or header.endswith("_ratio"):
                    cell.number_format = "0.00%"
        worksheet.row_dimensions[1].height = 30
    workbook.save(WORKBOOK_PATH)


def _sheet_purpose(sheet_name: str) -> str:
    readable = sheet_name.split("_", 1)[-1].replace("_", " ").lower()
    return f"Pipeline output table: {readable}"


def log_step(step: str, sheets: dict[str, pd.DataFrame], note: str = "") -> None:
    prepared = {name[:31]: _excel_frame(frame) for name, frame in sheets.items()}
    timestamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
    for sheet_name, frame in prepared.items():
        existing = SHEET_LOG.get(sheet_name)
        SHEET_LOG[sheet_name] = {
            "sheet_sequence": existing["sheet_sequence"] if existing else len(SHEET_LOG) + 1,
            "worksheet_name": sheet_name,
            "pipeline_step": step,
            "data_rows": len(frame),
            "data_columns": len(frame.columns),
            "purpose": _sheet_purpose(sheet_name),
            "last_written_utc": timestamp,
        }
    SHEET_LOG["00_Run_Log"].update(
        {
            "data_rows": len(SHEET_LOG),
            "last_written_utc": timestamp,
        }
    )
    run_log = pd.DataFrame(SHEET_LOG.values()).sort_values("sheet_sequence")
    frames = {"00_Run_Log": run_log, **prepared}
    mode = "a" if WORKBOOK_PATH.exists() else "w"
    options = {"engine": "openpyxl", "mode": mode}
    if mode == "a":
        options["if_sheet_exists"] = "replace"
    with pd.ExcelWriter(WORKBOOK_PATH, **options) as writer:
        for sheet_name, frame in frames.items():
            frame.to_excel(writer, sheet_name=sheet_name, index=False)
    _style_sheets(list(frames))
    print(
        f"Logged {step}: {len(prepared)} worksheet(s); "
        f"{len(SHEET_LOG)} physical worksheets recorded"
    )


guide = pd.DataFrame(
    [
        {"setting": "input_directory", "value": str(INPUT_DIRECTORY), "meaning": "Folder containing the three Sample 2 JSON files"},
        {"setting": "workbook", "value": str(WORKBOOK_PATH), "meaning": "Audit workbook updated after every completed step"},
        {"setting": "run_llm", "value": RUN_LLM, "meaning": "False avoids the 14 paid narrative calls"},
        {"setting": "model", "value": MODEL, "meaning": "OpenAI model used when RUN_LLM is True"},
    ]
)
log_step("00_runtime_setup", {"00_Run_Guide": guide})
display(guide)

Logged 00_runtime_setup: 1 worksheet(s); 2 physical worksheets recorded


,setting,value,meaning
0,input_directory,/Users/abdelmoo/Desktop/CAPI Analysis/Team2-Ma...,Folder containing the three Sample 2 JSON files
1,workbook,/Users/abdelmoo/Desktop/CAPI Analysis/Team2-Ma...,Audit workbook updated after every completed step
2,run_llm,False,False avoids the 14 paid narrative calls
3,model,gpt-5-mini,OpenAI model used when RUN_LLM is True


## Step 1 - Load the raw cycle

The loader checks that all three required files exist and have the expected
top-level JSON type. The inventory check confirms that the Meta object contains
campaign, adset, ad, creative, and insight collections.

In [2]:
from src_2.ingestion import load_sample2
from src_2.ingestion.data_quality import inventory

raw_payload = load_sample2(INPUT_DIRECTORY)
source_inventory = inventory(raw_payload)
inventory_frame = pd.DataFrame(
    [{"collection": key, "record_count": value} for key, value in vars(source_inventory).items()]
)
source_files = pd.DataFrame(
    [
        {"file": name, "path": str(INPUT_DIRECTORY / name), "size_bytes": (INPUT_DIRECTORY / name).stat().st_size}
        for name in ("meta_data.json", "conversations.json", "products.json")
    ]
)
log_step(
    "01_load_raw_json",
    {"01_Source_Inventory": inventory_frame, "01_Source_Files": source_files},
    "No business calculations are performed in this step.",
)
display(inventory_frame)

Logged 01_load_raw_json: 2 worksheet(s); 4 physical worksheets recorded


,collection,record_count
0,campaigns,12
1,adsets,26
2,ads,40
3,creatives,30
4,insights,1243
5,conversations,788
6,products,105


## Step 2 - Normalize canonical dimensions and facts

IDs are standardized as strings, dates become timestamps, numeric media fields
become numbers, and duplicates are removed. WhatsApp rows are linked to the Meta
hierarchy through source IDs and the ad lookup.

Organic and direct conversations remain in the canonical conversation table, but
have no paid campaign ID. They will therefore be excluded from paid scorecards.
Raw message text, phone numbers, and customer names are not copied into canonical
data or Excel.

In [3]:
from src_2.ingestion import normalize_cycle

canonical = normalize_cycle(raw_payload)
source_context = (
    canonical.conversations.groupby("source_platform", dropna=False)
    .agg(
        conversations=("conversation_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        delivered_orders=("is_delivered", "sum"),
        net_revenue=("net_revenue", "sum"),
    )
    .reset_index()
)
canonical_inventory = pd.DataFrame(
    [
        {"canonical_table": name, "rows": len(getattr(canonical, name)), "natural_grain": grain}
        for name, grain in {
            "campaigns": "one row per campaign",
            "adsets": "one row per adset",
            "ads": "one row per ad",
            "creatives": "one row per creative",
            "media_daily": "one row per ad and date",
            "conversations": "one row per conversation",
            "order_lines": "one row per ordered product line",
            "products": "one row per product",
        }.items()
    ]
)
log_step(
    "02_normalize_canonical_data",
    {
        "02_Canonical_Inventory": canonical_inventory,
        "02_Source_Context": source_context,
        "02_Campaigns": canonical.campaigns,
        "02_Adsets": canonical.adsets,
        "02_Ads": canonical.ads,
        "02_Creatives": canonical.creatives,
        "02_Media_Daily": canonical.media_daily,
        "02_Conversations": canonical.conversations,
        "02_Order_Lines": canonical.order_lines,
        "02_Products": canonical.products,
    },
    "Organic/direct are retained for context and excluded later when entity IDs are missing.",
)
display(canonical_inventory)
display(source_context)

Logged 02_normalize_canonical_data: 10 worksheet(s); 14 physical worksheets recorded


,canonical_table,rows,natural_grain
0,campaigns,12,one row per campaign
1,adsets,26,one row per adset
2,ads,40,one row per ad
3,creatives,30,one row per creative
4,media_daily,1243,one row per ad and date
5,conversations,788,one row per conversation
6,order_lines,1081,one row per ordered product line
7,products,105,one row per product


,source_platform,conversations,unique_customers,delivered_orders,net_revenue
0,direct,72,69,44,62060.0
1,meta_ctwa,617,531,331,457574.0
2,organic,99,95,49,59196.0


## Step 3 - Assess evidence quality

This compares Meta-attributed conversation starts with supplied Meta-sourced
WhatsApp conversations and checks whether their campaign/adset/ad IDs resolve.
The ratio is a reconciliation warning, not proven literal coverage, because the
two sources may use different event definitions, windows, and deduplication.

In [4]:
from src_2.ingestion import build_data_quality_report

data_quality = build_data_quality_report(canonical)
quality_frame = pd.DataFrame([data_quality.model_dump(mode="json")]).drop(columns=["warnings"])
quality_warnings = pd.DataFrame({"warning": data_quality.warnings})
log_step(
    "03_build_data_quality_report",
    {"03_Data_Quality": quality_frame, "03_Quality_Warnings": quality_warnings},
    "The current Sample 2 evidence status is limited_evidence.",
)
display(quality_frame)
display(quality_warnings)

Logged 03_build_data_quality_report: 2 worksheet(s); 16 physical worksheets recorded


,status,meta_conversation_starts,observed_meta_whatsapp_conversations,reconciliation_ratio,event_definitions_reconciled,unmatched_campaigns,unmatched_adsets,unmatched_ads
0,limited_evidence,116098,617,0.005314,False,0,0,0


,warning
0,Meta-attributed conversation starts and suppli...


## Step 4 - Load business rules

Campaign-type YAML selects the business job, primary KPI, supporting KPIs,
guardrails, and one allocation metric. Budget YAML defines the illustrative
100-unit allocation policy. Pydantic rejects missing, invalid, or extra fields.

In [5]:
from src_2.infrastructure.configuration import load_budget_policy, load_campaign_type_registry

campaign_registry = load_campaign_type_registry()
budget_policy = load_budget_policy()
campaign_rules = []
for campaign_type, rule in campaign_registry.campaign_types.items():
    campaign_rules.append(
        {
            "campaign_type": campaign_type.value,
            "business_job": rule.business_job,
            "success_question": rule.success_question,
            "primary_kpis": ", ".join(rule.primary_kpis),
            "supporting_kpis": ", ".join(rule.supporting_kpis),
            "allocation_metric": rule.allocation_metric.metric,
            "allocation_direction": rule.allocation_metric.direction,
            "budget_pool": rule.budget_pool.value,
            "guardrails": [guard.model_dump(mode="json") for guard in rule.guardrails],
        }
    )
campaign_rules_frame = pd.DataFrame(campaign_rules)
budget_policy_frame = pd.json_normalize(budget_policy.model_dump(mode="json"), sep=".")
action_eligibility = pd.DataFrame(
    [
        {"budget_pool": pool, "eligible_action": action}
        for pool, actions in budget_policy.action_eligibility.items()
        for action in actions
    ]
)
log_step(
    "04_load_business_configuration",
    {
        "04_Campaign_Rules": campaign_rules_frame,
        "04_Budget_Policy": budget_policy_frame,
        "04_Action_Eligibility": action_eligibility,
    },
    "The current test envelope follows historical experimental spend, not a fixed 30% reserve.",
)
display(campaign_rules_frame)

Logged 04_load_business_configuration: 3 worksheet(s); 19 physical worksheets recorded


,campaign_type,business_job,success_question,primary_kpis,supporting_kpis,allocation_metric,allocation_direction,budget_pool,guardrails
0,awareness,Create efficient qualified visibility before e...,Did the campaign reach people efficiently with...,reach,"cpm, frequency, link_ctr_pct, meta_conversatio...",cpm,lower,core,"[{'metric': 'frequency', 'operator': 'within',..."
1,always_on,Acquire steady sales with stable economics.,Did the campaign generate continuous delivered...,net_roas,"delivered_rate, cost_per_delivered_order, obse...",net_roas,higher,core,"[{'metric': 'observed_conversations', 'operato..."
2,promotional,Convert offer-driven WhatsApp interest into de...,Did the promotion create valuable delivered or...,net_roas,"order_creation_rate, delivered_rate, aov, cost...",net_roas,higher,core,"[{'metric': 'observed_conversations', 'operato..."
3,seasonal,Capture time-sensitive demand during a limited...,Did the campaign generate strong delivered rev...,net_revenue_per_day,"delivered_orders, net_roas, cost_per_delivered...",net_revenue_per_day,higher,core,"[{'metric': 'observed_conversations', 'operato..."
4,experimental,Generate enough evidence to identify audiences...,Did the test produce enough comparable outcome...,observed_conversations,"unique_creatives, ad_count, cost_per_observed_...",cost_per_observed_conversation,lower,test,"[{'metric': 'unique_creatives', 'operator': 'g..."
5,launch,Validate demand for newly launched or newly em...,Did the launch create product adoption and del...,net_revenue,"unique_products_ordered, net_revenue_per_day, ...",net_revenue,higher,core,"[{'metric': 'observed_conversations', 'operato..."
6,scale,Increase outcome volume while protecting effic...,"Did volume increase without damaging return, d...",observed_conversations,"observed_conversations_per_day, net_roas, cost...",observed_conversations,higher,core,"[{'metric': 'observed_conversations', 'operato..."
7,retention,Re-engage existing customers and create profit...,Did previous customers return and complete val...,repeat_delivered_orders,"repeat_conversation_rate, repeat_order_rate, n...",repeat_delivered_orders,higher,core,"[{'metric': 'observed_conversations', 'operato..."


## Step 5 - Build five scorecards

Media, conversations, and product lines are aggregated independently at each
entity level and only then merged. Ratios are recomputed from aggregated totals;
daily ratios are not averaged.

In [6]:
from src_2.analytics import build_scorecards

raw_scorecards = build_scorecards(canonical)
reconciliation_rows = []
for level in ("campaign", "adset", "ad", "creative", "audience"):
    frame = raw_scorecards.by_level(level)
    reconciliation_rows.append(
        {
            "level": level,
            "entities": len(frame),
            "spend": frame["spend"].sum(),
            "observed_conversations": frame["observed_conversations"].sum(),
            "net_revenue": frame["net_revenue"].sum(),
        }
    )
reconciliation = pd.DataFrame(reconciliation_rows)
log_step(
    "05_build_scorecards",
    {
        "05_Reconciliation": reconciliation,
        "05_Campaign_Scorecard": raw_scorecards.campaign,
        "05_Adset_Scorecard": raw_scorecards.adset,
        "05_Ad_Scorecard": raw_scorecards.ad,
        "05_Creative_Scorecard": raw_scorecards.creative,
        "05_Audience_Scorecard": raw_scorecards.audience,
    },
    "Spend and attributed outcomes reconcile across every scorecard level.",
)
display(reconciliation)
display(raw_scorecards.campaign.head())

Logged 05_build_scorecards: 6 worksheet(s); 25 physical worksheets recorded


,level,entities,spend,observed_conversations,net_revenue
0,campaign,12,402274.64,617,457574.0
1,adset,26,402274.64,617,457574.0
2,ad,40,402274.64,617,457574.0
3,creative,31,402274.64,617,457574.0
4,audience,22,402274.64,617,457574.0


,campaign_id,campaign_name,campaign_type,objective,entity_status,entity_effective_status,start_date,end_date,entity_id,entity_name,...,delivered_rate,refund_rate,negative_outcome_rate,repeat_conversation_rate,repeat_order_rate,net_revenue_per_day,observed_conversations_per_day,delivered_rate_pct,negative_outcome_rate_pct,order_creation_rate_pct
0,120209876543220001,Always-On Premium Acquisition,always_on,OUTCOME_SALES,ACTIVE,ACTIVE,2025-12-30,2026-06-27,120209876543220001,Always-On Premium Acquisition,...,0.468468,0.161290,0.468468,0.396396,0.403846,365.688889,0.616667,46.846847,46.846847,63.063063
1,120209876543220002,Awareness Boost January,awareness,OUTCOME_AWARENESS,COMPLETED,COMPLETED,2026-01-15,2026-02-14,120209876543220002,Awareness Boost January,...,0.250000,0.000000,0.750000,0.000000,0.000000,35.354839,0.129032,25.000000,75.000000,50.000000
2,120209876543220003,Pre-Ramadan Bundle Promo,promotional,OUTCOME_LEADS,COMPLETED,COMPLETED,2026-02-01,2026-02-27,120209876543220003,Pre-Ramadan Bundle Promo,...,0.456522,0.160000,0.478261,0.000000,0.000000,1070.555556,1.703704,45.652174,47.826087,69.565217
3,120209876543220004,January Trial Bundle Test,experimental,OUTCOME_ENGAGEMENT,COMPLETED,COMPLETED,2026-01-05,2026-01-31,120209876543220004,January Trial Bundle Test,...,0.333333,0.285714,0.666667,0.000000,0.000000,111.259259,0.555556,33.333333,66.666667,53.333333
4,120209876543220005,Ramadan Suhoor Specials,seasonal,OUTCOME_SALES,COMPLETED,COMPLETED,2026-02-28,2026-03-29,120209876543220005,Ramadan Suhoor Specials,...,0.640625,0.046512,0.328125,1.000000,1.000000,1266.733333,2.133333,64.062500,32.812500,75.000000


## Step 6 - Document calculated fields

These definitions explain the calculated columns already present in every
scorecard. `net_revenue` and `net_roas` are contribution proxies, not profit,
because product cost, fulfillment, tax, and overhead are unavailable.

In [7]:
kpi_definitions = pd.DataFrame(
    [
        {"metric": "frequency", "calculation": "impressions / reach", "business_meaning": "Average exposure per reached person"},
        {"metric": "link_ctr_pct", "calculation": "link_clicks / impressions * 100", "business_meaning": "Share of impressions producing a link click"},
        {"metric": "cpm", "calculation": "spend / impressions * 1000", "business_meaning": "Cost per 1,000 impressions"},
        {"metric": "cpc", "calculation": "spend / link_clicks", "business_meaning": "Cost per link click"},
        {"metric": "cost_per_observed_conversation", "calculation": "spend / observed WhatsApp conversations", "business_meaning": "Media cost per supplied attributed conversation"},
        {"metric": "cost_per_delivered_order", "calculation": "spend / delivered orders", "business_meaning": "Media cost per delivered order"},
        {"metric": "net_roas", "calculation": "net revenue / spend", "business_meaning": "Observed revenue return per media unit spent"},
        {"metric": "aov", "calculation": "delivered revenue / delivered orders", "business_meaning": "Average delivered order value"},
        {"metric": "delivered_rate", "calculation": "delivered orders / observed conversations", "business_meaning": "Share of supplied conversations becoming delivered orders"},
        {"metric": "negative_outcome_rate", "calculation": "ghosted + cancelled + refunded + adversarial / observed conversations", "business_meaning": "Share of observed conversations ending negatively"},
        {"metric": "net_revenue_per_day", "calculation": "net revenue / active media days", "business_meaning": "Revenue normalized for different campaign durations"},
        {"metric": "repeat_order_rate", "calculation": "repeat delivered orders / delivered orders", "business_meaning": "Delivered-order share attributed to repeat cycles"},
    ]
)
log_step(
    "06_document_kpi_definitions",
    {"06_KPI_Definitions": kpi_definitions},
    "All ratios use aggregated numerators and denominators.",
)
display(kpi_definitions)

Logged 06_document_kpi_definitions: 1 worksheet(s); 26 physical worksheets recorded


,metric,calculation,business_meaning
0,frequency,impressions / reach,Average exposure per reached person
1,link_ctr_pct,link_clicks / impressions * 100,Share of impressions producing a link click
2,cpm,spend / impressions * 1000,"Cost per 1,000 impressions"
3,cpc,spend / link_clicks,Cost per link click
4,cost_per_observed_conversation,spend / observed WhatsApp conversations,Media cost per supplied attributed conversation
5,cost_per_delivered_order,spend / delivered orders,Media cost per delivered order
6,net_roas,net revenue / spend,Observed revenue return per media unit spent
7,aov,delivered revenue / delivered orders,Average delivered order value
8,delivered_rate,delivered orders / observed conversations,Share of supplied conversations becoming deliv...
9,negative_outcome_rate,ghosted + cancelled + refunded + adversarial /...,Share of observed conversations ending negatively


## Step 7 - Build campaign evidence packs

Each pack is the exact handoff from measurement to the decision and campaign
analysis stages. It contains campaign-type context, KPI evidence, benchmarks,
guardrails, limitations, and child-entity evidence.

The current benchmark resolver excludes the entity being assessed, then uses a
current-cycle same-type median and falls back to a current-cycle portfolio median.
Historical observations and approved campaign targets are not yet loaded.

In [8]:
from src_2.analytics import build_evidence_packs
from src_2.application.reporting import _manifest

manifest = _manifest(canonical)
evidence_packs = build_evidence_packs(
    manifest.cycle_id, raw_scorecards, campaign_registry, data_quality
)
pack_summary = []
metric_evidence_rows = []
child_evidence_rows = []
for pack in evidence_packs:
    pack_summary.append(
        {
            "campaign_id": pack.campaign_id,
            "campaign_name": pack.campaign_name,
            "campaign_type": pack.campaign_type.value,
            "business_job": pack.business_job,
            "success_question": pack.success_question,
            "evidence_status": pack.evidence_status.value,
            "adsets": len(pack.adsets),
            "ads": len(pack.ads),
            "creatives": len(pack.creatives),
            "audiences": len(pack.audiences),
        }
    )
    for category, metrics in (
        ("primary", pack.primary_kpis),
        ("supporting", pack.supporting_kpis),
        ("guardrail", pack.guardrails),
        ("allocation", [pack.allocation_kpi] if pack.allocation_kpi else []),
    ):
        for metric in metrics:
            metric_evidence_rows.append(
                {"campaign_id": pack.campaign_id, "category": category, **metric.model_dump(mode="json")}
            )
    for entities in (pack.adsets, pack.ads, pack.creatives, pack.audiences):
        for entity in entities:
            for metric in entity.metrics:
                child_evidence_rows.append(
                    {
                        "campaign_id": pack.campaign_id,
                        "entity_level": entity.entity_level.value,
                        "entity_id": entity.entity_id,
                        "entity_name": entity.entity_name,
                        "metric": metric.metric,
                        "actual": metric.actual,
                        "benchmark": metric.benchmark,
                        "passed": metric.passed,
                        "evidence_count": metric.evidence_count,
                    }
                )
pack_summary_frame = pd.DataFrame(pack_summary)
metric_evidence_frame = pd.DataFrame(metric_evidence_rows)
child_evidence_frame = pd.DataFrame(child_evidence_rows)
log_step(
    "07_build_evidence_packs",
    {
        "07_Evidence_Packs": pack_summary_frame,
        "07_Campaign_Evidence": metric_evidence_frame,
        "07_Child_Evidence": child_evidence_frame,
    },
    "One typed CampaignEvidencePack is created for each campaign.",
)
display(pack_summary_frame)
display(metric_evidence_frame.head(20))

Logged 07_build_evidence_packs: 3 worksheet(s); 29 physical worksheets recorded


,campaign_id,campaign_name,campaign_type,business_job,success_question,evidence_status,adsets,ads,creatives,audiences
0,120209876543220001,Always-On Premium Acquisition,always_on,Acquire steady sales with stable economics.,Did the campaign generate continuous delivered...,limited_evidence,3,6,5,2
1,120209876543220002,Awareness Boost January,awareness,Create efficient qualified visibility before e...,Did the campaign reach people efficiently with...,limited_evidence,1,1,1,1
2,120209876543220003,Pre-Ramadan Bundle Promo,promotional,Convert offer-driven WhatsApp interest into de...,Did the promotion create valuable delivered or...,limited_evidence,2,3,2,2
3,120209876543220004,January Trial Bundle Test,experimental,Generate enough evidence to identify audiences...,Did the test produce enough comparable outcome...,limited_evidence,1,1,1,1
4,120209876543220005,Ramadan Suhoor Specials,seasonal,Capture time-sensitive demand during a limited...,Did the campaign generate strong delivered rev...,limited_evidence,2,4,3,2
5,120209876543220006,Ramadan Iftar Premium Bundles,seasonal,Capture time-sensitive demand during a limited...,Did the campaign generate strong delivered rev...,limited_evidence,3,5,3,3
6,120209876543220007,Eid Gifting Premium,seasonal,Capture time-sensitive demand during a limited...,Did the campaign generate strong delivered rev...,limited_evidence,2,3,2,2
7,120209876543220008,Post-Eid Lookalike Test,experimental,Generate enough evidence to identify audiences...,Did the test produce enough comparable outcome...,limited_evidence,2,2,2,1
8,120209876543220009,Summer Premium Launch,launch,Validate demand for newly launched or newly em...,Did the launch create product adoption and del...,limited_evidence,3,6,5,2
9,120209876543220010,Lookalike Scale Cycle 3,scale,Increase outcome volume while protecting effic...,"Did volume increase without damaging return, d...",limited_evidence,2,3,3,1


,campaign_id,category,metric,label,actual,benchmark,benchmark_source,direction,passed,evidence_count
0,120209876543220001,primary,net_roas,Net return on ad spend,0.467553,1.205624e+00,current-cycle portfolio median,higher,False,111
1,120209876543220001,supporting,delivered_rate,Delivered-order rate,0.468468,5.000000e-01,current-cycle portfolio median,higher,False,111
2,120209876543220001,supporting,cost_per_delivered_order,Cost per delivered order,2707.382885,8.776245e+02,current-cycle portfolio median,lower,False,111
3,120209876543220001,supporting,observed_conversations,Observed WhatsApp conversations,111.000000,4.600000e+01,current-cycle portfolio median,higher,True,111
4,120209876543220001,supporting,net_revenue_per_day,Net revenue per active day,365.688889,1.070556e+03,current-cycle portfolio median,higher,False,111
5,120209876543220001,guardrail,observed_conversations,Observed WhatsApp conversations,111.000000,1.000000e+01,configured threshold,higher,True,111
6,120209876543220001,guardrail,delivered_orders,Delivered orders,52.000000,0.000000e+00,configured threshold,higher,True,111
7,120209876543220001,guardrail,negative_outcome_rate,Negative-outcome rate,0.468468,4.047619e-01,current-cycle portfolio median,lower,False,111
8,120209876543220001,allocation,net_roas,Net return on ad spend,0.467553,1.205624e+00,current-cycle portfolio median,higher,False,111
9,120209876543220002,primary,reach,Reach,636884.000000,2.598221e+06,current-cycle portfolio median,higher,False,4


## Step 8 - Assign campaign target and funding decisions

The deterministic assessor answers two different questions:

- `target_status`: did the campaign perform its campaign-type business job?
- `next_cycle_action`: should the campaign scale, remain a test, receive no
  funding, or wait for more evidence?

Limited evidence prevents an automatic scale action.

In [9]:
from src_2.analytics import DeterministicCampaignAssessor

assessor = DeterministicCampaignAssessor()
assessments = [
    assessor.assess(pack, campaign_registry.campaign_types[pack.campaign_type])
    for pack in evidence_packs
]
assessment_frame = pd.DataFrame(
    [
        {
            "campaign_id": item.campaign_id,
            "campaign_name": item.campaign_name,
            "campaign_type": item.campaign_type.value,
            "target_status": item.target_status.value,
            "next_cycle_action": item.next_cycle_action.value,
            "evidence_status": item.evidence_status.value,
            "reason_codes": item.reason_codes,
        }
        for item in assessments
    ]
)
assessment_details = pd.DataFrame(
    [
        {
            "campaign_id": item.campaign_id,
            "result_type": result_type,
            **metric.model_dump(mode="json"),
        }
        for item in assessments
        for result_type, results in (("primary", item.primary_results), ("guardrail", item.guardrail_results))
        for metric in results
    ]
)
log_step(
    "08_assess_campaigns",
    {"08_Campaign_Decisions": assessment_frame, "08_Decision_Details": assessment_details},
    "Funding actions are deterministic and independent from the LLM narrative.",
)
display(assessment_frame)

Logged 08_assess_campaigns: 2 worksheet(s); 31 physical worksheets recorded


,campaign_id,campaign_name,campaign_type,target_status,next_cycle_action,evidence_status,reason_codes
0,120209876543220001,Always-On Premium Acquisition,always_on,not_achieved,do_not_fund,limited_evidence,"[limited_evidence, primary:net_roas:fail, guar..."
1,120209876543220002,Awareness Boost January,awareness,not_achieved,keep_as_test,limited_evidence,"[limited_evidence, primary:reach:fail]"
2,120209876543220003,Pre-Ramadan Bundle Promo,promotional,achieved_with_concerns,keep_as_test,limited_evidence,"[limited_evidence, primary:net_roas:pass, guar..."
3,120209876543220004,January Trial Bundle Test,experimental,not_achieved,keep_as_test,limited_evidence,"[limited_evidence, primary:observed_conversati..."
4,120209876543220005,Ramadan Suhoor Specials,seasonal,not_achieved,keep_as_test,limited_evidence,"[limited_evidence, primary:net_revenue_per_day..."
5,120209876543220006,Ramadan Iftar Premium Bundles,seasonal,not_achieved,do_not_fund,limited_evidence,"[limited_evidence, primary:net_revenue_per_day..."
6,120209876543220007,Eid Gifting Premium,seasonal,achieved_with_concerns,keep_as_test,limited_evidence,"[limited_evidence, primary:net_revenue_per_day..."
7,120209876543220008,Post-Eid Lookalike Test,experimental,achieved_with_concerns,keep_as_test,limited_evidence,"[limited_evidence, primary:observed_conversati..."
8,120209876543220009,Summer Premium Launch,launch,achieved_with_concerns,keep_as_test,limited_evidence,"[limited_evidence, primary:net_revenue:pass]"
9,120209876543220010,Lookalike Scale Cycle 3,scale,not_achieved,keep_as_test,limited_evidence,"[limited_evidence, primary:observed_conversati..."


## Step 9 - Add child-entity decisions

Adsets, ads, creatives, and audiences are compared with peers inside their parent
campaign using the campaign type's allocation metric. A blocked parent blocks its
children. Minimum observed-conversation thresholds are 5 for adsets/audiences and
3 for ads/creatives.

In [10]:
from src_2.analytics import enrich_scorecards

scorecards = enrich_scorecards(
    raw_scorecards, assessments, campaign_registry, data_quality
)
child_columns = [
    "campaign_id", "campaign_name", "entity_id", "entity_name",
    "spend", "observed_conversations", "delivered_orders", "net_revenue",
    "allocation_metric", "allocation_metric_value", "allocation_benchmark",
    "allocation_metric_passed", "evidence_status", "next_cycle_action", "decision_reason",
]
child_decision_sheets = {}
child_summary_rows = []
for number, level in enumerate(("adset", "ad", "creative", "audience"), start=1):
    frame = scorecards.by_level(level)
    selected = frame[[column for column in child_columns if column in frame]].copy()
    child_decision_sheets[f"09_{level.title()}_Decisions"] = selected
    for action, count in selected["next_cycle_action"].value_counts().items():
        child_summary_rows.append({"entity_level": level, "next_cycle_action": action, "entities": count})
child_summary = pd.DataFrame(child_summary_rows)
child_decision_sheets["09_Child_Summary"] = child_summary
log_step(
    "09_enrich_child_decisions",
    child_decision_sheets,
    "Children receive actions and reasons, but the current allocator does not assign child budget units.",
)
display(child_summary)

Logged 09_enrich_child_decisions: 5 worksheet(s); 36 physical worksheets recorded


,entity_level,next_cycle_action,entities
0,adset,do_not_fund,17
1,adset,keep_as_test,8
2,adset,insufficient_evidence,1
3,ad,do_not_fund,28
4,ad,keep_as_test,11
5,ad,insufficient_evidence,1
6,creative,do_not_fund,20
7,creative,keep_as_test,10
8,creative,insufficient_evidence,1
9,audience,do_not_fund,15


## Step 10 - Build the normalized budget scenario

The allocator starts with 100 illustrative units, preserves each campaign type's
historical spend share, filters campaigns by deterministic action eligibility,
and distributes each type envelope using one configured allocation KPI.

Money blocked by the rules remains unallocated. This output is not an approved
currency budget and is non-operational while evidence is limited.

In [11]:
from src_2.analytics import DeterministicBudgetAllocator

allocator = DeterministicBudgetAllocator()
budget_scenario = allocator.allocate(
    manifest.cycle_id,
    scorecards.campaign,
    assessments,
    campaign_registry,
    budget_policy,
    data_quality,
)
budget_summary = pd.DataFrame(
    [
        {
            "cycle_id": budget_scenario.cycle_id,
            "scenario_name": budget_scenario.scenario_name,
            "total_budget_units": budget_scenario.total_budget_units,
            "allocated_units": sum(item.budget_units for item in budget_scenario.allocations),
            "unallocated_units": budget_scenario.unallocated_units,
            "evidence_status": budget_scenario.evidence_status.value,
            "operational": budget_scenario.operational,
        }
    ]
)
budget_allocations = pd.DataFrame(
    [
        {
            **item.model_dump(mode="json"),
            "entity_level": item.entity_level.value,
            "action": item.action.value,
        }
        for item in budget_scenario.allocations
    ]
).sort_values("budget_units", ascending=False)
budget_assumptions = pd.DataFrame({"assumption": budget_scenario.assumptions})
log_step(
    "10_allocate_normalized_budget",
    {
        "10_Budget_Summary": budget_summary,
        "10_Budget_Allocations": budget_allocations,
        "10_Budget_Assumptions": budget_assumptions,
    },
    "The current Sample 2 scenario remains illustrative and non-operational.",
)
display(budget_summary)
display(budget_allocations[["entity_name", "action", "budget_units", "reason"]])

Logged 10_allocate_normalized_budget: 3 worksheet(s); 39 physical worksheets recorded


,cycle_id,scenario_name,total_budget_units,allocated_units,unallocated_units,evidence_status,operational
0,cycle_2025-12-30_2026-06-27,POC normalized next-cycle allocation,100.0,62.764106,37.235894,limited_evidence,False


,entity_name,action,budget_units,reason
6,Eid Gifting Premium,keep_as_test,20.074536,Receives 77.4% of the seasonal envelope using ...
8,Summer Premium Launch,keep_as_test,17.051992,Receives 100.0% of the launch envelope using n...
9,Lookalike Scale Cycle 3,keep_as_test,7.821987,Receives 100.0% of the scale envelope using ob...
4,Ramadan Suhoor Specials,keep_as_test,5.869153,Receives 22.6% of the seasonal envelope using ...
2,Pre-Ramadan Bundle Promo,keep_as_test,4.762869,Receives 60.8% of the promotional envelope usi...
10,Mid-Year Sale,keep_as_test,3.070389,Receives 39.2% of the promotional envelope usi...
7,Post-Eid Lookalike Test,keep_as_test,1.588187,Receives 60.4% of the experimental envelope us...
1,Awareness Boost January,keep_as_test,1.483230,Receives 100.0% of the awareness envelope usin...
3,January Trial Bundle Test,keep_as_test,1.041763,Receives 39.6% of the experimental envelope us...
0,Always-On Premium Acquisition,do_not_fund,0.000000,No units assigned because the deterministic ac...


## Step 11 - Optional structured LLM narrative

When enabled, the notebook performs:

1. One campaign-analysis call per campaign: `CampaignEvidencePack` -> `CampaignInsight`.
2. One portfolio call: assessments plus campaign insights -> `PortfolioInsight`.
3. One narrator call: portfolio insight plus fixed budget -> `StakeholderReport`.

Pydantic validates every response. The LLM explains supplied evidence and fixed
decisions; it does not recalculate KPIs, target statuses, actions, or budget.

In [12]:
from src_2.intelligence import OpenAICampaignAnalyst, OpenAIPortfolioSynthesizer, OpenAIReportNarrator

llm_input_summary = pd.DataFrame(
    [
        {
            "campaign_id": pack.campaign_id,
            "campaign_name": pack.campaign_name,
            "campaign_type": pack.campaign_type.value,
            "evidence_status": pack.evidence_status.value,
            "primary_kpis": ", ".join(metric.metric for metric in pack.primary_kpis),
            "guardrails": ", ".join(metric.metric for metric in pack.guardrails),
            "adsets": len(pack.adsets),
            "ads": len(pack.ads),
            "creatives": len(pack.creatives),
            "audiences": len(pack.audiences),
        }
        for pack in evidence_packs
    ]
)

campaign_insights = []
portfolio_insight = None
stakeholder_report = None
llm_sheets = {"11_LLM_Input_Summary": llm_input_summary}

if RUN_LLM:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("RUN_LLM is True but OPENAI_API_KEY is missing from .env")
    analyst = OpenAICampaignAnalyst(model=MODEL)
    campaign_insights = [analyst.analyze(pack) for pack in evidence_packs]
    portfolio_insight = OpenAIPortfolioSynthesizer(model=MODEL).synthesize(
        assessments, campaign_insights
    )
    stakeholder_report = OpenAIReportNarrator(model=MODEL).narrate(
        portfolio_insight, budget_scenario
    )
    llm_status = pd.DataFrame([{"run_llm": True, "model": MODEL, "api_calls": len(evidence_packs) + 2, "status": "completed"}])
    llm_sheets["11_Campaign_Insights"] = pd.DataFrame(
        [item.model_dump(mode="json") for item in campaign_insights]
    )
    llm_sheets["11_Portfolio_Insight"] = pd.DataFrame(
        [portfolio_insight.model_dump(mode="json")]
    )
    llm_sheets["11_Stakeholder_Report"] = pd.DataFrame(
        [stakeholder_report.model_dump(mode="json")]
    )
else:
    llm_status = pd.DataFrame(
        [{"run_llm": False, "model": MODEL, "api_calls": 0, "status": "skipped", "reason": "Set RUN_LLM=True to execute the 14-call narrative pipeline."}]
    )
llm_sheets["11_LLM_Status"] = llm_status
log_step(
    "11_optional_llm_narrative",
    llm_sheets,
    "LLM execution is optional; all KPI, decision, and budget outputs already exist deterministically.",
)
display(llm_status)

Logged 11_optional_llm_narrative: 2 worksheet(s); 41 physical worksheets recorded


,run_llm,model,api_calls,status,reason
0,False,gpt-5-mini,0,skipped,Set RUN_LLM=True to execute the 14-call narrat...


## Step 12 - Assemble final outputs

The audit workbook is always complete through the deterministic budget stage.
When the LLM section is enabled, this cell also assembles the same
`CompletedCycleReport` used by Streamlit and exports its structured JSON.

In [13]:
from src_2.application.reporting import CompletedCycleReport

report = None
final_summary = pd.DataFrame(
    [
        {
            "cycle_id": manifest.cycle_id,
            "reporting_start": manifest.reporting_start,
            "reporting_end": manifest.reporting_end,
            "campaigns": len(scorecards.campaign),
            "campaigns_keep_as_test": sum(item.next_cycle_action.value == "keep_as_test" for item in assessments),
            "campaigns_do_not_fund": sum(item.next_cycle_action.value == "do_not_fund" for item in assessments),
            "allocated_budget_units": sum(item.budget_units for item in budget_scenario.allocations),
            "unallocated_budget_units": budget_scenario.unallocated_units,
            "operational": budget_scenario.operational,
            "llm_report_generated": stakeholder_report is not None,
            "excel_sheets_logged": len(SHEET_LOG) + 1,
            "excel_workbook": str(WORKBOOK_PATH),
        }
    ]
)

final_sheets = {"12_Final_Summary": final_summary}
if stakeholder_report is not None:
    report = CompletedCycleReport(
        manifest=manifest,
        data_quality=data_quality,
        canonical_data=canonical,
        scorecards=scorecards,
        evidence_packs=evidence_packs,
        assessments=assessments,
        insights=campaign_insights,
        portfolio_insight=portfolio_insight,
        budget_scenario=budget_scenario,
        stakeholder_report=stakeholder_report,
    )
    FINAL_JSON_PATH.write_text(
        json.dumps(report.to_export_dict(), indent=2, ensure_ascii=True),
        encoding="utf-8",
    )
    final_sheets["12_Final_Artifacts"] = pd.DataFrame(
        [{"artifact": "completed_cycle_report_json", "path": str(FINAL_JSON_PATH)}]
    )

log_step(
    "12_finalize_outputs",
    final_sheets,
    "The Excel workbook contains an auditable snapshot of every completed stage.",
)
workbook_sheet_names = load_workbook(WORKBOOK_PATH, read_only=True).sheetnames
actual_sheet_log = pd.DataFrame(SHEET_LOG.values()).sort_values("sheet_sequence")
assert len(workbook_sheet_names) == len(actual_sheet_log)
assert set(workbook_sheet_names) == set(actual_sheet_log["worksheet_name"])
display(final_summary)
display(actual_sheet_log)
print(f"Audit workbook: {WORKBOOK_PATH}")
print(f"Actual worksheets logged: {len(actual_sheet_log)}")
if report is None:
    print("Structured report JSON was not generated because RUN_LLM=False.")
else:
    print(f"Structured report JSON: {FINAL_JSON_PATH}")

Logged 12_finalize_outputs: 1 worksheet(s); 42 physical worksheets recorded


,cycle_id,reporting_start,reporting_end,campaigns,campaigns_keep_as_test,campaigns_do_not_fund,allocated_budget_units,unallocated_budget_units,operational,llm_report_generated,excel_sheets_logged,excel_workbook
0,cycle_2025-12-30_2026-06-27,2025-12-30,2026-06-27,12,9,3,62.764106,37.235894,False,False,42,/Users/abdelmoo/Desktop/CAPI Analysis/Team2-Ma...


,sheet_sequence,worksheet_name,pipeline_step,data_rows,data_columns,purpose,last_written_utc
0,1,00_Run_Log,00_runtime_setup,42,7,Index of every physical worksheet written to t...,2026-08-05T14:37:16+00:00
1,2,00_Run_Guide,00_runtime_setup,4,3,Pipeline output table: run guide,2026-08-05T14:36:27+00:00
2,3,01_Source_Inventory,01_load_raw_json,7,2,Pipeline output table: source inventory,2026-08-05T14:36:27+00:00
3,4,01_Source_Files,01_load_raw_json,3,3,Pipeline output table: source files,2026-08-05T14:36:27+00:00
4,5,02_Canonical_Inventory,02_normalize_canonical_data,8,3,Pipeline output table: canonical inventory,2026-08-05T14:36:28+00:00
5,6,02_Source_Context,02_normalize_canonical_data,3,5,Pipeline output table: source context,2026-08-05T14:36:28+00:00
6,7,02_Campaigns,02_normalize_canonical_data,12,8,Pipeline output table: campaigns,2026-08-05T14:36:28+00:00
7,8,02_Adsets,02_normalize_canonical_data,26,21,Pipeline output table: adsets,2026-08-05T14:36:28+00:00
8,9,02_Ads,02_normalize_canonical_data,40,9,Pipeline output table: ads,2026-08-05T14:36:28+00:00
9,10,02_Creatives,02_normalize_canonical_data,30,12,Pipeline output table: creatives,2026-08-05T14:36:28+00:00


Audit workbook: /Users/abdelmoo/Desktop/CAPI Analysis/Team2-MarketingExpert/outputs/sample2_pipeline_audit/sample2_pipeline_step_log.xlsx
Actual worksheets logged: 42
Structured report JSON was not generated because RUN_LLM=False.


## How to use the workbook

Start with `00_Run_Log`, then follow sheets by numeric prefix. The most important
business sheets are:

- `03_Data_Quality`: whether recommendations can be operational.
- `05_Campaign_Scorecard`: campaign economics and outcomes.
- `07_Campaign_Evidence`: benchmarks and pass/fail evidence.
- `08_Campaign_Decisions`: target achievement and next-cycle action.
- `09_*_Decisions`: adset, ad, creative, and audience actions.
- `10_Budget_Allocations`: illustrative next-cycle campaign budget.
- `11_*`: optional LLM input and output audit.

The workbook is a trace of Python-calculated outputs. Business assumptions remain
visible in YAML and are copied into the configuration and budget sheets.